# MedVS-AI — Export a real ISIC case bundle (Phase 1, option 1)

Downloads ISIC 2018 Task 1 and packages ~120 cases (image + lesion mask + benign/malignant label)
into `case_bundle.zip` for the web app's drawing game.

**No GPU needed.** `Runtime > Run all`. The image zip is ~10 GB (fast on Colab). Then locally:
```
cd web/backend
python load_bundle.py ~/Downloads/case_bundle.zip
python seed_cases.py --bundle ../data/bundle
```
> NOT FOR CLINICAL USE.

## 1. Download ISIC 2018 Task 1

In [ ]:
import os, zipfile, urllib.request
IMAGES_URL = "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task1-2_Training_Input.zip"
MASKS_URL  = "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task1_Training_GroundTruth.zip"
IMAGES_DIR = "data/ISIC2018_Task1-2_Training_Input"
MASKS_DIR  = "data/ISIC2018_Task1_Training_GroundTruth"
os.makedirs("data", exist_ok=True)

def _hook(b, bs, t):
    if t > 0 and b % 500 == 0: print(f"  {min(100, b*bs*100//t)}%", end="\r")

def fetch(url, zname, outdir):
    if os.path.isdir(outdir) and os.listdir(outdir): print("have", outdir); return
    if not os.path.exists(zname): print("downloading", url); urllib.request.urlretrieve(url, zname, _hook)
    print("\nextracting", zname)
    with zipfile.ZipFile(zname) as z: z.extractall("data")

fetch(MASKS_URL, "data/masks.zip", MASKS_DIR)
fetch(IMAGES_URL, "data/images.zip", IMAGES_DIR)
print("images:", len(os.listdir(IMAGES_DIR)), "| masks:", len(os.listdir(MASKS_DIR)))


## 2. Build the bundle (image + GT mask + benign/malignant label from the ISIC API)

In [ ]:
import re, csv, json, zipfile, shutil, random, urllib.request
from PIL import Image

N_CASES, MAX_TRIES, SEED = 120, 600, 42
OUT = "bundle"
if os.path.isdir(OUT): shutil.rmtree(OUT)
os.makedirs(f"{OUT}/images"); os.makedirs(f"{OUT}/gt")

ISIC_RE = re.compile(r"(ISIC_\d+)")
pairs = []
for fn in sorted(os.listdir(MASKS_DIR)):
    if not fn.endswith(".png"): continue
    cid = ISIC_RE.search(fn).group(1)
    ip = os.path.join(IMAGES_DIR, cid + ".jpg")
    if os.path.exists(ip): pairs.append((cid, ip, os.path.join(MASKS_DIR, fn)))
random.Random(SEED).shuffle(pairs)

def label(cid):
    # ISIC API now uses diagnosis_1 = "Benign"/"Malignant"/"Indeterminate" (no more benign_malignant)
    try:
        with urllib.request.urlopen(f"https://api.isic-archive.com/api/v2/images/{cid}/", timeout=15) as r:
            meta = json.loads(r.read())
        d1 = ((meta.get("metadata", {}).get("clinical", {}) or {}).get("diagnosis_1") or "").lower()
        return d1 if d1 in ("benign", "malignant") else None
    except Exception:
        return None

rows, tried = [], 0
for cid, ip, mp in pairs:
    if len(rows) >= N_CASES or tried >= MAX_TRIES: break
    tried += 1
    lab = label(cid)
    if lab not in ("benign", "malignant"): continue
    Image.open(ip).convert("RGB").resize((256, 256)).save(f"{OUT}/images/{cid}.png")
    Image.open(mp).convert("L").resize((256, 256), Image.NEAREST).save(f"{OUT}/gt/{cid}.png")
    rows.append({"case_id": cid, "benign_malignant": lab})
    if len(rows) % 20 == 0: print(f"  collected {len(rows)} (tried {tried})")

with open(f"{OUT}/labels.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["case_id", "benign_malignant"]); w.writeheader(); w.writerows(rows)
with zipfile.ZipFile("case_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            p = os.path.join(root, fn); z.write(p, os.path.relpath(p, "."))
assert rows, "Collected 0 labeled cases — check the ISIC API is reachable from this runtime."
print(f"exported {len(rows)} labeled cases ({sum(r['benign_malignant']=='malignant' for r in rows)} malignant) -> case_bundle.zip")
from google.colab import files
files.download("case_bundle.zip")


## Done
Download `case_bundle.zip`, then locally:
```
cd web/backend
python load_bundle.py ~/Downloads/case_bundle.zip
python seed_cases.py --bundle ../data/bundle
python app.py
```